## Figure 2 - Estimate pleural pressure

In [ ]:
# env: notebook
import numpy as np
import shutil
import os
from pathlib import Path

import dolfin_mech as dmech

from Reader_MeshDisplacement import MeshDisplacementReader
from PleuralPressure_Maps import PleuralPressureMaps

In [2]:
%load_ext autoreload
%autoreload 2

### Download data

In [ ]:
# TODO Update links to zenodo

In [ ]:
!curl -L -o results_ante_pro1.zip "https://sdrive.cnrs.fr/s/bMBFcWQF4QmSqDD/download?path=%2F&files=Results_ANTE_PRO1"
!unzip -o results_ante_pro1.zip
!rm -f results_ante_pro1.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  282M    0  282M    0     0  14.1M      0 --:--:--  0:00:19 --:--:-- 17.6M--:--:--  0:00:06 --:--:-- 8682k
Archive:  results_ante_pro1.zip
 extracting: .//Results_ANTE_PRO1/230804_02GA06/Fields_from_image_analysis/mesh_LL.xdmf  
 extracting: .//Results_ANTE_PRO1/230804_02GA06/Fields_from_image_analysis/mesh_RL.xdmf  
 extracting: .//Results_ANTE_PRO1/230804_02GA06/Fields_from_image_analysis/mesh_LL.h5  
 extracting: .//Results_ANTE_PRO1/230804_02GA06/Fields_from_image_analysis/mesh_RL.h5  
 extracting: .//Results_ANTE_PRO1/230804_02GA06/Mesh/Mesh_LL_00.xdmf  
 extracting: .//Results_ANTE_PRO1/230804_02GA06/Mesh/Mesh_LL_00.h5  
 extracting: .//Results_ANTE_PRO1/230804_02GA06/Mesh/Volume_vs_time.pkl  
 extracting: .//Results_ANTE_PRO1/230804_02GA06/Me

In [ ]:
!curl -L -o results_ante_sup2.zip "https://sdrive.cnrs.fr/s/bMBFcWQF4QmSqDD/download?path=%2F&files=Results_ANTE_SUP2"
!unzip -o results_ante_sup2.zip
!rm -f results_ante_sup2.zip

### Parameters

#### Material

In [3]:
alpha_lst = [0.16] # [0.016, 0.16, 1.6] # kPa
params = {
    # "alpha":0.16,     # kPa
    "gamma":0.5,      # [-]
    "c1":0.4,         # kPa From Peyraut & Genet (2024), Table 1
    "c2":0.2,         # kPa From Peyraut & Genet (2024), Table 1
    "kappa":1e2,      # kPa
    "eta":1e-5,       # KPa
    "rho_solid":1e-6} # g/mm3
mat_params = {"scaling":"linear", "parameters":params}

#### Loading

In [4]:
pe = -0.5    # kPa
g  = +9.81e3 # mm/s2

### Computing responses

In [5]:
acquisition_lst = []
acquisition_lst += ["ANTE_SUP2"]
acquisition_lst += ["ANTE_PRO1"]

volunteers_lst  = []
volunteers_lst += ["230526_02BT01"]
volunteers_lst += ["230704_02JY02"]
volunteers_lst += ["230706_02LH03"]
volunteers_lst += ["230728_02CL05"]
volunteers_lst += ["230804_02GA06"]
volunteers_lst += ["230929_02JD07"]
volunteers_lst += ["231013_02BF08"]
volunteers_lst += ["231020_02AR09"]
volunteers_lst += ["231024_02DL10"]
volunteers_lst += ["231027_02AS11"]
volunteers_lst += ["231110_02CC12"]
volunteers_lst += ["231117_02HC13"]
volunteers_lst += ["231124_02VG14"]
volunteers_lst += ["231205_02NS15"]
volunteers_lst += ["231219_02YF16"]
volunteers_lst += ["231222_02OC17"]
volunteers_lst += ["240105_02MD18"]
volunteers_lst += ["240109_02AL19"]
volunteers_lst += ["240119_02JN20"]
volunteers_lst += ["240202_02CH21"]
volunteers_lst += ["240206_02HA04"]
volunteers_lst += ["240524_02TA22"]
volunteers_lst += ["241011_02NB26"]
volunteers_lst += ["241018_02ZT27"]
volunteers_lst += ["241210_02JL29"]
volunteers_lst += ["241217_02JC30"]
volunteers_lst += ["250128_02RP32"]
volunteers_lst += ["250204_02MH35"]
volunteers_lst += ["250207_02SN36"]
volunteers_lst += ["250218_02TG23"]
volunteers_lst += ["250221_02VL38"]
volunteers_lst += ["250304_02WD39"]
volunteers_lst += ["250307_02MC41"]
volunteers_lst += ["250513_02WM46"]
volunteers_lst += ["250909_02LB54"]
volunteers_lst += ["250923_02FM56"]
volunteers_lst += ["250926_02AH57"]
volunteers_lst += ["251014_02RM58"]
volunteers_lst += ["251107_02SL59"]
volunteers_lst += ["251118_02JM60"]
volunteers_lst += ["251128_02LS24"]
volunteers_lst += ["251219_02GD67"]

# Repeated volunteers
# volunteers_lst += ["230718_02HA04"]
# volunteers_lst += ["240827_02TG23"]
# volunteers_lst += ["240903_02LS24"]

region_lst = []
region_lst += ["LL"]
region_lst += ["RL"]

# exclude specific cases
exclude_lst  = []
exclude_lst += ["ANTE_PRO1-231222_02OC17"]
exclude_lst += ["ANTE_PRO2-231222_02OC17"]
exclude_lst += ["POST_PRO1-231222_02OC17"]
exclude_lst += ["ANTE_SUP2-250307_02MC41"]
exclude_lst += ["ANTE_PRO2-250307_02MC41"]
exclude_lst += ["ANTE_SUP2-251014_02RM58"]
exclude_lst += ["ANTE_PRO1-251014_02RM58"]
exclude_lst += ["ANTE_PRO2-251014_02RM58"]
exclude_lst += ["POST_PRO1-251014_02RM58"]
exclude_lst += ["POST_SUP1-251014_02RM58"]
exclude_lst += ["ANTE_PRO1-231013_02BF08-LL"] # mask not correct

n_phases = 32

In [6]:
gravity_lst = [1] # [0,1]
error_lst   = []
np.random.seed(0)

for acquisition in acquisition_lst:
    if acquisition in ["ANTE_SUP1", "ANTE_SUP2", "POST_SUP1"]:
        position = "supine"
    elif acquisition in ["ANTE_PRO1", "ANTE_PRO2", "POST_PRO1"]:
        position = "prone"
    else:
        raise ValueError(f"Unknown position for acquisition {acquisition}")
    
    for volunteer in volunteers_lst:
        if f"{acquisition}-{volunteer}" in exclude_lst:
            continue

        for region in region_lst:
            if f"{acquisition}-{volunteer}-{region}" in exclude_lst:
                continue
            
            print(f"Processing {acquisition}, {volunteer}, {region} region")

            try:
                for gravity_ in gravity_lst:
                    if position == "supine":
                        gravity_ = gravity_ * 1
                    elif position == "prone":
                        gravity_ = gravity_ * -1

                    for alpha_   in alpha_lst:
                        ### Remove previous results
                        resultsPath = f"./Results_{acquisition}/{volunteer}"
                        p_plPath    = f"{resultsPath}/Pleural_pressure_estimation"

                        os.makedirs(p_plPath, exist_ok=True)

                        for file in os.listdir(p_plPath):
                            if f"{region}_alpha{alpha_}_gravity{gravity_}_pe{pe}" in file:
                                file_path = os.path.join(p_plPath, file)
                                if os.path.isfile(file_path):
                                    os.remove(file_path)
                                elif os.path.isdir(file_path):
                                    shutil.rmtree(file_path)

                        figures_folder = os.path.join(p_plPath, "Figures")
                        if os.path.exists(figures_folder):
                            for file in os.listdir(figures_folder):
                                if f"{region}_alpha{alpha_}_gravity{gravity_}_pe{pe}" in file:
                                    file_path = os.path.join(figures_folder, file)
                                    if os.path.isfile(file_path):
                                        os.remove(file_path)
                                    elif os.path.isdir(file_path):
                                        shutil.rmtree(file_path)


                        ### Run simulation
                        MDR = MeshDisplacementReader(acquisition, volunteer, region, n_phases)
                        cube_params = {"path_and_file_name": MDR.mesh_exhal_file}
                        
                        mat_params["parameters"]["alpha"] = alpha_
                        
                        ### Compute unloaded configuration
                        print ('Computing unloaded configuration...')
                        # A uniform porosity Phis0 is assumed in the unloaded configuration.
                        n_cells                = MDR.mesh_exhal.cells().shape[0]
                        Phis0_unloaded_imposed = [np.random.uniform(low=0.4, high=0.6) for i in range(n_cells)]

                        ### Fixed point algorithm
                        # phis_exhal_old = [np.random.uniform(low=0.4, high=0.6) for i in range(n_cells)]
                        # error = 1.0
                        # print ("Iterating to find unloaded configuration...")
                        # while error > 1e-2:
                        #     U_exhal_to_unloaded, Phis0_unloaded, dV_exhal = dmech.run_RivlinCube_PoroHyperelasticity(
                        #         inverse         = 1,
                        #         cube_params     = cube_params,
                        #         porosity_params = {"type":"function_xml_from_array", "val":phis_exhal_old},
                        #         mat_params      = mat_params,
                        #         inertia_params  = {"applied":True, "rho_val":1e-8},
                        #         step_params     = {"dt_min":1e-4, "dt_ini":0.25},
                        #         load_params     = {"type":"p_boundary_condition0", "f":gravity_*g, "f_direction":"x", "P0":float(pe)},
                        #         res_basename    = f"{resultsPath}/Pleural_pressure_estimation/Unloaded_{region}_alpha{alpha_}_gravity{gravity_}_pe{pe}",
                        #         get_results     = 1,
                        #         verbose         = 1)

                        #     ### computing the end-exhalation configuration
                        #     U_unloaded_to_exhal, phis_exhal, dV_unloaded = dmech.run_RivlinCube_PoroHyperelasticity(
                        #         inverse         = 0,
                        #         cube_params     = cube_params,
                        #         move_params     = {"move":True, "U":U_exhal_to_unloaded},
                        #         porosity_params = {"type":"function_xml_from_array", "val":Phis0_unloaded_imposed},
                        #         mat_params      = mat_params,
                        #         inertia_params  = {"applied":True, "rho_val":1e-8},
                        #         step_params     = {"dt_min":1e-4, "dt_ini":0.25}, 
                        #         load_params     = {"type":"p_boundary_condition", "f":gravity_*g, "f_direction":"x", "P0":float(pe)},
                        #         res_basename    = f"{resultsPath}/Pleural_pressure_estimation/Direct-exhal_{region}_alpha{alpha_}_gravity{gravity_}_pe{pe}",
                        #         get_results     = 1,
                        #         verbose         = 1)

                        #     error = np.linalg.norm(phis_exhal - phis_exhal_old)
                        #     print (f"Error in phis_exhal: {error:.4f}")

                        #     phis_exhal_old = phis_exhal

                        ### Generalized Poromechanics
                        U_exhal_to_unloaded, phis_exhal, dV_exhal = dmech.run_RivlinCube_PoroHyperelasticity(
                            inverse         = 1,
                            cube_params     = cube_params,
                            porosity_params = {"known":"Phis0", "type":"function_xml_from_array", "val":Phis0_unloaded_imposed},
                            mat_params      = mat_params,
                            inertia_params  = {"applied":True, "rho_val":1e-8},
                            step_params     = {"dt_min":1e-4, "dt_ini":0.25},
                            load_params     = {"type":"p_boundary_condition0", "f":gravity_*g, "f_direction":"x", "P0":float(pe)},
                            res_basename    = f"{resultsPath}/Pleural_pressure_estimation/Unloaded_{region}_alpha{alpha_}_gravity{gravity_}_pe{pe}",
                            get_results     = 1,
                            verbose         = 0)

                        ### Compute stress
                        print ('Computing pleural pressure maps...')
                        mesh_unloaded = MDR.read_mesh(MDR.mesh_exhal_file,
                                                      U_move=[U_exhal_to_unloaded],
                                                      save_name=f"{resultsPath}/Pleural_pressure_estimation/mesh_unloaded_{region}_alpha{alpha_}_gravity{gravity_}_pe{pe}.xdmf")

                        PPM = PleuralPressureMaps(acquisition, volunteer, region, alpha_, gravity_, pe, mat_params["parameters"], mesh_unloaded, U_exhal_to_unloaded, MDR, np.array(Phis0_unloaded_imposed))
                        PPM.save_boundary_traction()

            except Exception as e:
                print(f"[ERROR]: {e}")
                error_lst.append((acquisition, volunteer, region, str(e)))


Processing ANTE_SUP2, 230526_02BT01, LL region
Computing unloaded configuration...
Computing pleural pressure maps...
Processing ANTE_SUP2, 230526_02BT01, RL region
Computing unloaded configuration...
Computing pleural pressure maps...
Processing ANTE_SUP2, 230704_02JY02, LL region
Computing unloaded configuration...
Computing pleural pressure maps...
Processing ANTE_SUP2, 230704_02JY02, RL region
Computing unloaded configuration...
Computing pleural pressure maps...
Processing ANTE_SUP2, 230706_02LH03, LL region
Computing unloaded configuration...
Computing pleural pressure maps...
Processing ANTE_SUP2, 230706_02LH03, RL region
Computing unloaded configuration...
Computing pleural pressure maps...
Processing ANTE_SUP2, 230728_02CL05, LL region
Computing unloaded configuration...
Computing pleural pressure maps...
Processing ANTE_SUP2, 230728_02CL05, RL region
Computing unloaded configuration...
Computing pleural pressure maps...
Processing ANTE_SUP2, 230804_02GA06, LL region
Computing

In [7]:
error_lst

[]